# Call Volume Patterns

This notebook aims to explore and inform on the trends, patterns, and phenomenons in the SPD Call Volume. The majority of the analysis is on the number of calls in the data over time (from 2009 to 2026) and is then further broken down by `call priority` and `event group type`. There are several seasonal periods within the data which were revealed by examining different frequencies. A basic outline of the sections is below:
- Loading Data, Basic Inspection
- Monthly Call Volume
- Daily Call Volume
- Hourly Call Volume

### Loading Data, Basic inspection

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import re
import statsmodels.api as sm
import statsmodels.formula.api as smf
import scipy.stats as stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from patsy import build_design_matrices
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.oneway import anova_oneway
from itertools import combinations
from scipy.stats import studentized_range
from scipy import stats
from statsmodels.tsa.seasonal import STL
from spd_snapshot import load_spd_call_snapshot
from spd_eda import summarize_spd_calls
from spd_event_volume import load_spd_event_volume
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "notebook"
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')


df, metadata = load_spd_call_snapshot(
    PROJECT_ROOT / "data" / "processed"
)

display(df.head())
summary = summarize_spd_calls(df)
print(summary)

Checking all columns are present

In [ ]:
TIME_COLUMN = "cad_event_original_time_queued"
EVENT_ID_COLUMN = "cad_event_number"
ROW_ID_COLUMN = "call_sign_dispatch_id"

required_columns = [
    TIME_COLUMN,
    EVENT_ID_COLUMN,
    ROW_ID_COLUMN,
    "event_group",
    "priority",
    "call_type",
    "dispatch_precinct",
    "dispatch_neighborhood",
]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

missing_columns

Checking Row Grain

In [ ]:
calls = df.copy()

calls[TIME_COLUMN] = pd.to_datetime(
    calls[TIME_COLUMN],
    errors="coerce",
)

display(calls[[TIME_COLUMN, EVENT_ID_COLUMN, ROW_ID_COLUMN]].head())
date_min = calls[TIME_COLUMN].min()
date_max = calls[TIME_COLUMN].max()

print(f"Start date of calls: {date_min} \nEnd date: {date_max}")

record_count = len(calls)
unique_call_events = calls[EVENT_ID_COLUMN].nunique()
unique_dispatch_records = calls[ROW_ID_COLUMN].nunique()

print(f"Number of records: {record_count} \nNumber of unique dispatch records: {unique_dispatch_records} \nNumber of unique call events: {unique_call_events}")

#### Row Grain Note

The data set is not one row per CAD event. `cad_event_number` identifies a CAD call/event, while `call_sign_dispatch_id` identifies a row-level dispatch record. Therefore, call volume should generally be measured using unique `cad_event_number`, while row counts represent dispatch/unit-response activity.

Making Date features

In [ ]:
valid_time = calls.dropna(subset=[TIME_COLUMN]).copy()

valid_time["date"] = valid_time[TIME_COLUMN].dt.date
valid_time["month"] = valid_time[TIME_COLUMN].dt.to_period("M").dt.to_timestamp()
valid_time["day_of_week"] = valid_time[TIME_COLUMN].dt.day_name()
valid_time["day_of_week_num"] = valid_time[TIME_COLUMN].dt.dayofweek
valid_time["hour"] = valid_time[TIME_COLUMN].dt.hour
valid_time["is_weekend"] = valid_time["day_of_week_num"].isin([5, 6])

display(valid_time[
    [
        TIME_COLUMN,
        "date",
        "month",
        "day_of_week",
        "day_of_week_num",
        "hour",
        "is_weekend",
    ]
].head())

print(f"Null counts: \n\n{valid_time[["date", "month", "day_of_week", "hour"]].isna().sum()}")

### Monthly Call Volume

In this section we examine the volume of calls each month from 2009 to 2026, first by total volume then by volume by event group, volume by call type, and volume by neighborhood

In [ ]:
#monthly_volume = load_spd_event_volume(
#    grain="month",
#    group_by=None,
#)
monthly_volume = pd.read_parquet((PROJECT_ROOT / 'data' / 'processed' / 'monthly_volume.parquet').as_posix())
monthly_volume = monthly_volume[monthly_volume['period_start'] != '2026-07-01']
monthly_volume.tail()

In [ ]:
fig = px.line(
    monthly_volume,
    x="period_start",
    y="event_count",
    markers=False,
    title="SPD Unique CAD Event Volume by Month",
    labels={
        "period_start": "Month",
        "event_count": "Unique CAD Events",
    },
)

fig.update_layout(
    hovermode="x unified",
    xaxis_title="Month",
    yaxis_title="Unique CAD Events",
)

# TAKE THIS OUT IF YOU DON"T LIKE IT DARK
fig.update_layout(
    template="plotly_dark",
    plot_bgcolor="#545455",   
    paper_bgcolor="#111111",
    )

def add_annotation(fig, df, date, text, ax=40, ay=-60):
    date = pd.to_datetime(date)

    row = df.loc[df["period_start"] == date]

    if row.empty:
        # Finds closest month if exact date is not present
        closest_idx = (df["period_start"] - date).abs().idxmin()
        row = df.loc[[closest_idx]]

    x_val = row["period_start"].iloc[0]
    y_val = row["event_count"].iloc[0]

    fig.add_annotation(
        x=x_val,
        y=y_val,
        text=text,
        showarrow=True,
        arrowhead=2,
        arrowsize=1,
        arrowwidth=1.5,
        ax=ax,
        ay=ay,
        bgcolor="rgba(17,17,17,0.85)",
        bordercolor="white",
        borderwidth=1,
        font=dict(
            size=12,
            color="white",
        ),
    )


# Example annotations — change these dates/texts
add_annotation(
    fig,
    monthly_volume,
    date="2020-03-01",
    text="COVID-era shift begins",
    ax=50,
    ay=-70,
)

add_annotation(
    fig,
    monthly_volume,
    date="2014-07-04",
    text="Summer volume increase",
    ax=40,
    ay=-60,
)

add_annotation(
    fig,
    monthly_volume,
    date="2014-12-10",
    text="Lower winter volume",
    ax=-50,
    ay=60,
)




fig.show()

Above we can see the monthly call volume over the entire span of the data. Annotated are some interesting patterns and deviations:
- The volume of calls demonstrates a recurring seasonal pattern, peaks in the summer, troughs in the winter
- The onset of COVID-19 massively brought down the volume of calls to SPD
- In the period following the pandemic the volume of calls has returned to its regularl fluctuations but has not fully recovered to pre-pandemic levels 

In [ ]:
#monthly_by_event_group = load_spd_event_volume(
#    grain="month",
#    group_by="event_group",
#    verbose=True,
#)

monthly_by_event_group = pd.read_parquet((PROJECT_ROOT / 'data' / 'processed' / 'monthly_by_event_group.parquet').as_posix())
monthly_by_event_group = monthly_by_event_group[monthly_by_event_group['period_start'] != '2026-07-01']
monthly_by_event_group.tail()

In [ ]:
fig = px.line(
    monthly_by_event_group,
    x="period_start",
    y="event_count",
    color="event_group",
    markers=False,
    title="SPD Unique CAD Event Volume by Month and Event Group",
    labels={
        "period_start": "Month",
        "event_count": "Unique CAD Events",
        "event_group": "Event Group",
    },
)

fig.update_layout(
    hovermode="x unified",
    xaxis_title="Month",
    yaxis_title="Unique CAD Events",
    legend_title_text="Event Group",
)

# TAKE THIS OUT IF YOU DON"T LIKE IT DARK
fig.update_layout(
    template="plotly_dark",
    plot_bgcolor="#545455",   
    paper_bgcolor="#111111",
    )

visible_groups = [
    "traffic",
    "disturbance",
    "theft",
]

for trace in fig.data:
    if trace.name not in visible_groups:
        trace.visible = "legendonly"


fig.show()

The figure above has interactivity that allows for the analysis of each event group over time, and the comparison of multiple event groups. Although only three of the highest volume event groups are displayed by default, the rest can be displayed by clicking on the text below the `Event Group` legend sitauted on the right of the chart. It is also possible to view a series on its own by doubble clicking the text in the legend. In the default display we can see that disturbance and traffic calls are the most prevalent in the data, and theft has comparatively less call volume. 

Below there are charts for event groups with meaningful patterns and phenomenons (note that less significant event groups' plots have been placed in the `Additional Visuals, Notes, Findings` section)

In [ ]:
theft_df = monthly_by_event_group[
    monthly_by_event_group["event_group"].isin(["theft"])
].copy()

fig_theft = px.line(
    theft_df,
    x="period_start",
    y="event_count",
    color="event_group",
    markers=False,
    title="Monthly SPD Unique CAD Event Volume: Theft",
    labels={
        "period_start": "Month",
        "event_count": "Unique CAD Events",
        "event_group": "Event Group",
    },
)

fig_theft.update_layout(
    hovermode="x unified",
    xaxis_title="Month",
    yaxis_title="Unique CAD Events",
    legend_title_text="Event Group",
    template="plotly_dark",
    plot_bgcolor="#545455",
    paper_bgcolor="#111111",
)

fig_theft.add_vline(
    x=pd.Timestamp("2020-06-01"),
    line_dash="dash",
    line_width=1,
    annotation_text="2020 shift",
    annotation_position="top left",
)

fig_theft.show()

Above we can see the theft related call volume over time in which we can see a downward trend in the period from 2009-2020 and then a period of relative stability from then onward.  

In [ ]:
assault_df = monthly_by_event_group[
    monthly_by_event_group["event_group"] == "assaults"
].copy()

fig_assault = px.line(
    assault_df,
    x="period_start",
    y="event_count",
    color="event_group",
    markers=False,
    title="Monthly SPD Unique CAD Event Volume: Assault",
    labels={
        "period_start": "Month",
        "event_count": "Unique CAD Events",
        "event_group": "Event Group",
    },
)

fig_assault.update_layout(
    hovermode="x unified",
    xaxis_title="Month",
    yaxis_title="Unique CAD Events",
    legend_title_text="Event Group",
    template="plotly_dark",
    plot_bgcolor="#545455",
    paper_bgcolor="#111111",
)

row_2014_jan = assault_df.loc[
    (assault_df["period_start"] - pd.Timestamp("2014-01-01")).abs().idxmin()
]

fig_assault.add_annotation(
    x=row_2014_jan["period_start"],
    y=row_2014_jan["event_count"],
    text="January 2014 spike worth investigating",
    showarrow=True,
    arrowhead=2,
    ax=50,
    ay=-80,
    bgcolor="rgba(17,17,17,0.85)",
    bordercolor="white",
    borderwidth=1,
    font=dict(size=11, color="white"),
)

row_2020_apr = assault_df.loc[
    (assault_df["period_start"] - pd.Timestamp("2020-04-01")).abs().idxmin()
]

fig_assault.add_annotation(
    x=row_2020_apr["period_start"],
    y=row_2020_apr["event_count"],
    text="2020 drop",
    showarrow=True,
    arrowhead=2,
    ax=50,
    ay=70,
    bgcolor="rgba(17,17,17,0.85)",
    bordercolor="white",
    borderwidth=1,
    font=dict(size=11, color="white"),
)

fig_assault.show()

Above we see a the assualt related call volume over time, in which there is a staggering jump around January of 2014. We can see that there is also a downward trend that stabilizes after 2020. 

In [ ]:
disturbances_df = monthly_by_event_group[
    monthly_by_event_group["event_group"].isin(["disturbance"])
].copy()

fig_disturbances = px.line(
    disturbances_df,
    x="period_start",
    y="event_count",
    color="event_group",
    markers=False,
    title="Monthly SPD Unique CAD Event Volume: Disturbances",
    labels={
        "period_start": "Month",
        "event_count": "Unique CAD Events",
        "event_group": "Event Group",
    },
)

fig_disturbances.update_layout(
    hovermode="x unified",
    xaxis_title="Month",
    yaxis_title="Unique CAD Events",
    legend_title_text="Event Group",
    template="plotly_dark",
    plot_bgcolor="#545455",
    paper_bgcolor="#111111",
)

fig_disturbances.add_vline(
    x=pd.Timestamp("2020-06-01"),
    line_dash="dash",
    line_width=1,
    annotation_text="2020 shock",
    annotation_position="top left",
)

row_2022_july = disturbances_df.loc[
    (disturbances_df["period_start"] - pd.Timestamp("2022-07-01")).abs().idxmin()
]

fig_disturbances.add_annotation(
    x=row_2022_july["period_start"],
    y=row_2022_july["event_count"],
    text="Recurring summer peak",
    showarrow=True,
    arrowhead=2,
    ax=50,
    ay=-70,
    bgcolor="rgba(17,17,17,0.85)",
    bordercolor="white",
    borderwidth=1,
    font=dict(size=11, color="white"),
)

fig_disturbances.show()

Above we can see the disturbance related call volume over time. There is of course a significant shock during 2020, but outside of that the only noticeable quality is the incredibly consistent seasonality (peaks in summer, troughs in winter).

In [ ]:
misdemeanor_df = monthly_by_event_group[
    monthly_by_event_group['event_group'] == 'misc. misdemeanors & violations'
    ].copy()

fig_misdemeanor = px.line(
    misdemeanor_df,
    x="period_start",
    y="event_count",
    color="event_group",
    markers=False,
    title="Monthly SPD Unique CAD Event Volume: Misdemeanors",
    labels={
        "period_start": "Month",
        "event_count": "Unique CAD Events",
        "event_group": "Event Group",
    },
)

fig_misdemeanor.update_layout(
    hovermode="x unified",
    xaxis_title="Month",
    yaxis_title="Unique CAD Events",
    legend_title_text="Event Group",
    template="plotly_dark",
    plot_bgcolor="#545455",
    paper_bgcolor="#111111",
)

row_2018_jan = misdemeanor_df.loc[
    (misdemeanor_df["period_start"] - pd.Timestamp("2018-01-01")).abs().idxmin()
]

fig_misdemeanor.add_annotation(
    x=row_2018_jan["period_start"],
    y=row_2018_jan["event_count"],
    text="Sharp reduction after 2018",
    showarrow=True,
    arrowhead=2,
    ax=50,
    ay=-70,
    bgcolor="rgba(17,17,17,0.85)",
    bordercolor="white",
    borderwidth=1,
    font=dict(size=11, color="white"),
)

fig_misdemeanor.show()

Above we can see the misdeameanor related call volume over time. In this plot we can notice a seasonal pattern in the period from 2009-2018 with the regular summer peaks we've been seeing in other event groups, but there is a very sharp reduction in the overall call volume after 2018. 

In [ ]:
narcotics_df = monthly_by_event_group[
    monthly_by_event_group["event_group"].isin(["narcotics"])
].copy()

fig_narcotics = px.line(
    narcotics_df,
    x="period_start",
    y="event_count",
    color="event_group",
    markers=False,
    title="Monthly SPD Unique CAD Event Volume: Narcotics",
    labels={
        "period_start": "Month",
        "event_count": "Unique CAD Events",
        "event_group": "Event Group",
    },
)

fig_narcotics.update_layout(
    hovermode="x unified",
    xaxis_title="Month",
    yaxis_title="Unique CAD Events",
    legend_title_text="Event Group",
    template="plotly_dark",
    plot_bgcolor="#545455",
    paper_bgcolor="#111111",
)

row_2020_june = narcotics_df.loc[
    (narcotics_df["period_start"] - pd.Timestamp("2020-06-01")).abs().idxmin()
]

fig_narcotics.add_annotation(
    x=row_2020_june["period_start"],
    y=row_2020_june["event_count"],
    text="Downward trend reverses after 2020",
    showarrow=True,
    arrowhead=2,
    ax=50,
    ay=-200,
    bgcolor="rgba(17,17,17,0.85)",
    bordercolor="white",
    borderwidth=1,
    font=dict(size=11, color="white"),
)

fig_narcotics.show()

Above we can see the narcotics related call volume over time. The period preceding 2020 demonstrated a downward trend in narcotics related calls, but after 2020 there has been a reversal of that trend. It is interesting to note that the relative peak seen during September of 2025 was followed by large drop-off (a drop of ~500 calls from September 2025 to February 2026).

In [ ]:
sex_offenses_df = monthly_by_event_group[
    monthly_by_event_group["event_group"].isin([
        "rape",
        "sex offenses (non-rape)",
    ])
].copy()

if sex_offenses_df.empty:
    raise ValueError("sex_offenses_df is empty. Check exact event_group names.")

fig_sex_offenses = px.line(
    sex_offenses_df,
    x="period_start",
    y="event_count",
    color="event_group",
    markers=False,
    title="Monthly SPD Unique CAD Event Volume: Rape and Sex Offenses",
    labels={
        "period_start": "Month",
        "event_count": "Unique CAD Events",
        "event_group": "Event Group",
    },
)

fig_sex_offenses.update_layout(
    hovermode="x unified",
    xaxis_title="Month",
    yaxis_title="Unique CAD Events",
    legend_title_text="Event Group",
    template="plotly_dark",
    plot_bgcolor="#545455",
    paper_bgcolor="#111111",
)

# Annotation for rape
rape_df = sex_offenses_df[
    sex_offenses_df["event_group"] == "rape"
].copy()

if rape_df.empty:
    raise ValueError("rape_df is empty. Exact event_group 'rape' was not found.")

row_rape = rape_df.loc[
    (rape_df["period_start"] - pd.Timestamp("2022-01-01")).abs().idxmin()
]

fig_sex_offenses.add_annotation(
    x=row_rape["period_start"],
    y=row_rape["event_count"],
    text="Rape calls fluctuate but may have slight upward trend",
    showarrow=True,
    arrowhead=2,
    ax=50,
    ay=-70,
    bgcolor="rgba(17,17,17,0.85)",
    bordercolor="white",
    borderwidth=1,
    font=dict(size=11, color="white"),
)

# Annotation for sex offenses non-rape
sex_non_rape_df = sex_offenses_df[
    sex_offenses_df["event_group"] == "sex offenses (non-rape)"
].copy()

if sex_non_rape_df.empty:
    raise ValueError(
        "sex_non_rape_df is empty. Exact event_group "
        "'sex offenses (non-rape)' was not found."
    )

row_sex_non_rape = sex_non_rape_df.loc[
    (sex_non_rape_df["period_start"] - pd.Timestamp("2014-08-01")).abs().idxmin()
]

fig_sex_offenses.add_annotation(
    x=row_sex_non_rape["period_start"],
    y=row_sex_non_rape["event_count"],
    text="Seasonality appears clearer before 2016",
    showarrow=True,
    arrowhead=2,
    ax=50,
    ay=-70,
    bgcolor="rgba(17,17,17,0.85)",
    bordercolor="white",
    borderwidth=1,
    font=dict(size=11, color="white"),
)

fig_sex_offenses.show()

Above we can see the sex offense related call volume. Some interesting things to note are the seasonality in non-rape sex offenses in the period preceeding 2016. During the period between 2009 and 2016 there was a strong pattern of non-rape sex offenses peaking in the summer and falling back down in the winter, but this pattern is no longer visually present. From 2016 onward the volume of sex offense calls has been on an upward trend. While inconclusive at this point there is some visual evidence that there is a slight upward trend in the rape related call volume, this is worth investigating further. It is also important to note that these are insights from the SPD call volume and not the statistics for the amount of these crimes that occur; many of these types of crimes do not get reported right away and this data does not fully represent the trends in the prevalence of these crimes in Seattle.

In [ ]:
traffic_df = monthly_by_event_group[
    monthly_by_event_group["event_group"].isin(["traffic"])
].copy()

fig_traffic = px.line(
    traffic_df,
    x="period_start",
    y="event_count",
    color="event_group",
    markers=False,
    title="Monthly SPD Unique CAD Event Volume: Traffic",
    labels={
        "period_start": "Month",
        "event_count": "Unique CAD Events",
        "event_group": "Event Group",
    },
)

fig_traffic.update_layout(
    hovermode="x unified",
    xaxis_title="Month",
    yaxis_title="Unique CAD Events",
    legend_title_text="Event Group",
    template="plotly_dark",
    plot_bgcolor="#545455",
    paper_bgcolor="#111111",
)

row_2020_apr = traffic_df.loc[
    (traffic_df["period_start"] - pd.Timestamp("2020-04-01")).abs().idxmin()
]

fig_traffic.add_annotation(
    x=row_2020_apr["period_start"],
    y=row_2020_apr["event_count"],
    text="Major 2020 drop",
    showarrow=True,
    arrowhead=2,
    ax=50,
    ay=70,
    bgcolor="rgba(17,17,17,0.85)",
    bordercolor="white",
    borderwidth=1,
    font=dict(size=11, color="white"),
)

row_2024_jan = traffic_df.loc[
    (traffic_df["period_start"] - pd.Timestamp("2026-06-01")).abs().idxmin()
]

fig_traffic.add_annotation(
    x=row_2024_jan["period_start"],
    y=row_2024_jan["event_count"],
    text="Still below pre-2020 levels but may recover",
    showarrow=True,
    arrowhead=2,
    ax=-160,
    ay=-70,
    bgcolor="rgba(17,17,17,0.85)",
    bordercolor="white",
    borderwidth=1,
    font=dict(size=11, color="white"),
)

fig_traffic.show()

In the above plot we can see the traffic related call volume over time, in which we can see the 2020 COVID-19 shock and an incomplete recovery in volume.

In [ ]:
vice_df = monthly_by_event_group[
    monthly_by_event_group["event_group"].isin(["vice"])
].copy()

fig_vice = px.line(
    vice_df,
    x="period_start",
    y="event_count",
    color="event_group",
    markers=False,
    title="Monthly SPD Unique CAD Event Volume: Vice",
    labels={
        "period_start": "Month",
        "event_count": "Unique CAD Events",
        "event_group": "Event Group",
    },
)

fig_vice.update_layout(
    hovermode="x unified",
    xaxis_title="Month",
    yaxis_title="Unique CAD Events",
    legend_title_text="Event Group",
    template="plotly_dark",
    plot_bgcolor="#545455",
    paper_bgcolor="#111111",
)

row_2020_june = vice_df.loc[
    (vice_df["period_start"] - pd.Timestamp("2020-06-01")).abs().idxmin()
]

fig_vice.add_annotation(
    x=row_2020_june["period_start"],
    y=row_2020_june["event_count"],
    text="Much lower than early 2010s volume",
    showarrow=True,
    arrowhead=2,
    ax=50,
    ay=-70,
    bgcolor="rgba(17,17,17,0.85)",
    bordercolor="white",
    borderwidth=1,
    font=dict(size=11, color="white"),
)

fig_vice.show()

In the above plot we can see the vice related call volume over time, which demonstrates a downturn in the call volume in the period available in the data (2009-2026) 

#### Volume by Call Type

In the below section we examine the volume of calls each month segmented by the type of call.

In [ ]:
#monthly_by_call_type = load_spd_event_volume(
#    grain="month",
#    group_by="call_type",
#    verbose=True,
#)

monthly_by_call_type = pd.read_parquet((PROJECT_ROOT / 'data' / 'processed' / 'monthly_by_call_type.parquet').as_posix())
monthly_by_call_type = monthly_by_call_type[monthly_by_call_type['period_start'] != '2026-07-01']
monthly_by_call_type.tail()


In [ ]:
top_call_types = (
    monthly_by_call_type
    .groupby("call_type")["event_count"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .index
)

monthly_by_call_type_top = monthly_by_call_type[
    monthly_by_call_type["call_type"].isin(top_call_types)
].copy()

fig = px.line(
    monthly_by_call_type_top,
    x="period_start",
    y="event_count",
    color="call_type",
    markers=False,
    title="SPD Unique CAD Event Volume by Month and Call Type: Top 10",
    labels={
        "period_start": "Month",
        "event_count": "Unique CAD Events",
        "call_type": "Call Type",
    },
)

fig.update_layout(
    hovermode="x unified",
    xaxis_title="Month",
    yaxis_title="Unique CAD Events",
    legend_title_text="Call Type",
)

# TAKE THIS OUT IF YOU DON"T LIKE IT DARK
fig.update_layout(
    template="plotly_dark",
    plot_bgcolor="#545455",   
    paper_bgcolor="#111111",
    )

visible_groups = [
    "911",
    "onview",
]

for trace in fig.data:
    if trace.name not in visible_groups:
        trace.visible = "legendonly"


fig.show()

Above we can see a plot of the two most common call types with the top ten present (the rest can be enabled by interacting with the chart). Within the plot we can see a sharp drop-off in the number of "onview" calls in the beginning of 2020. Within the last 6 years we have seen a climb in onview calls, but as it stands now the number of onview calls each month is at ~66% of what it was pre-COVID-19. It is worth noting that onview is defined as any event that was proactively responded to by the SPD officer upon witnessing the crime/event. 

#### Volume by Neighborhood 

In the below section we examine the volume of calls each month segmented by the neighborhood in which the calls took place.

In [ ]:
#monthly_by_neighborhood = load_spd_event_volume(
#    grain="month",
#    group_by="dispatch_neighborhood",
#    verbose=True,
#)

monthly_by_neighborhood = pd.read_parquet((PROJECT_ROOT / 'data' / 'processed' / 'monthly_by_neighborhood.parquet').as_posix())
print('Head of Monthly Call Volume by Neighborhood:  \n')
print(monthly_by_neighborhood.tail())
monthly_by_neighborhood.replace(r'\s*-\s*', np.nan, regex=True, inplace=True)
monthly_by_neighborhood.replace(r'\s*null\s*', np.nan, regex=True, inplace=True)
monthly_by_neighborhood.replace(r'\s*unknown\s*', np.nan, regex=True, inplace=True)
monthly_by_neighborhood = monthly_by_neighborhood.dropna(subset=['dispatch_neighborhood'])
print('Missing value number in Monthly Call Volume by Neighborhood (after cleaning):  \n')
print(monthly_by_neighborhood['dispatch_neighborhood'].isna().sum())
monthly_by_neighborhood = monthly_by_neighborhood[monthly_by_neighborhood['period_start'] != '2026-07-01']
print(monthly_by_neighborhood.tail())

In [ ]:
top_neighborhoods_monthly = (
    monthly_by_neighborhood
    .groupby("dispatch_neighborhood")["event_count"]
    .sum()
    .sort_values(ascending=False)
    .head(12)
    .index
)

monthly_by_neighborhood_top = monthly_by_neighborhood[
    monthly_by_neighborhood["dispatch_neighborhood"].isin(top_neighborhoods_monthly)
].copy()

monthly_by_neighborhood_top.head()

fig = px.line(
    monthly_by_neighborhood_top,
    x="period_start",
    y="event_count",
    color="dispatch_neighborhood",
    markers=False,
    title="Monthly SPD Unique CAD Event Volume by Dispatch Neighborhood: Top 12",
    labels={
        "period_start": "Month",
        "event_count": "Unique CAD Events",
        "dispatch_neighborhood": "Dispatch Neighborhood",
    },
)

fig.update_layout(
    hovermode="x unified",
    xaxis_title="Month",
    yaxis_title="Unique CAD Events",
    legend_title_text="Dispatch Neighborhood",
)

# TAKE THIS OUT IF YOU DON"T LIKE IT DARK
fig.update_layout(
    template="plotly_dark",
    plot_bgcolor="#545455",   
    paper_bgcolor="#111111",
    )

visible_groups = [
    "ballard south",
    "capitol hill",
    "downtown commercial",
]

for trace in fig.data:
    if trace.name not in visible_groups:
        trace.visible = "legendonly"


fig.show()

Above is a plot that shows the monthly call volume for each MCPP neighborhood, in it we can see that downtown commercial was typically the neighborhood with the highest call volume but has recently (~November 2024) been overtaken by capitol hill. The top 5 highest call volume neighborhoods are: 1. Capitol Hill 2. Downtown Commercial 3. North Gate 4. Chinatown/International District 5. Ballard South

### Daily Call Volume Analysis 

In the below section we examine the trends present in the data at a daily frequency rather than monthly frequency.

In [ ]:
daily_volume = (
    valid_time
    .groupby("date")
    .agg(
        unique_call_events=(EVENT_ID_COLUMN, "nunique"),
        dispatch_records=(ROW_ID_COLUMN, "nunique"),
    )
    .reset_index()
    .sort_values("date")
)

daily_volume["date"] = pd.to_datetime(daily_volume["date"])

daily_volume["unique_call_events_7d_avg"] = (
    daily_volume["unique_call_events"]
    .rolling(window=7, min_periods=1)
    .mean()
)

daily_volume.head()

fig = px.line(
    daily_volume,
    x="date",
    y=["unique_call_events", "unique_call_events_7d_avg"],
    title="Daily SPD Call Events with 7-Day Average",
    labels={
        "date": "Date",
        "value": "Unique call events",
        "variable": "Metric",
    },
)

# TAKE THIS OUT IF YOU DON"T LIKE IT DARK
fig.update_layout(
    template="plotly_dark",
    plot_bgcolor="#545455",   
    paper_bgcolor="#111111",
    )

fig.show()

Above we can see the daily call volume over the last year, we can see evidence of a 7 day seasonality where call volume is highest in the middle of the week. 

In [ ]:
# Create daily time series
s = (
    daily_volume
    .copy()
    .assign(date=lambda df: pd.to_datetime(df["date"]))
    .sort_values("date")
    .set_index("date")["unique_call_events"]
    .asfreq("D")
)

# Handle any missing days
s = s.interpolate(method="time").ffill().bfill()

# STL decomposition
# period=7 because this is daily data with weekly seasonality
stl = STL(
    s,
    period=7,
    robust=True
)

res = stl.fit()

# Plot STL components
#fig = res.plot()
#fig.set_size_inches(12, 8)
#fig.suptitle("STL Decomposition of Daily SPD Unique Call Events", y=1.02)

#plt.tight_layout()
#plt.show()

annotations = [
    {
        "component": "observed",
        "date": "2026-01-01",
        "text": "Original Series",
        "xytext": (-35, 50),
    },
    {
        "component": "trend",
        "date": "2025-12-25",
        "text": "Trend shows a trough during non-summer months",
        "xytext": (-116.75, 95),
    },
    {
        "component": "seasonal",
        "date": "2025-10-15",
        "text": "Weekly seasonality repeats throughout",
        "xytext": (-90, 30),
    },
    {
        "component": "resid",
        "date": "2025-12-25",
        "text": "Large residuals are uncommon, most hover around 0",
        "xytext": (30, 5),
    },
]
with plt.style.context("dark_background"):
    fig = res.plot()
    fig.set_size_inches(13, 9)
    fig.suptitle(
        "STL Decomposition of Daily SPD Unique Call Events",
        fontsize=16,
        y=1.02
    )

    axes = fig.axes

    component_axes = {
        "observed": axes[0],
        "trend": axes[1],
        "seasonal": axes[2],
        "resid": axes[3],
    }

    component_series = {
        "observed": res.observed,
        "trend": res.trend,
        "seasonal": res.seasonal,
        "resid": res.resid,
    }

    for ax in axes:
        ax.grid(True, alpha=0.25)

        for line in ax.lines:
            line.set_linewidth(1.8)

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    axes[0].set_ylabel("Observed")
    axes[1].set_ylabel("Trend")
    axes[2].set_ylabel("Seasonal")
    axes[3].set_ylabel("Residual")

    # Add annotations
    for note in annotations:
        component = note["component"]
        target_date = pd.Timestamp(note["date"])

        ax = component_axes[component]
        series = component_series[component].dropna()

        # Find nearest available date in the time series
        nearest_idx = series.index.get_indexer([target_date], method="nearest")[0]
        x = series.index[nearest_idx]
        y = series.iloc[nearest_idx]

        ax.scatter(
            x,
            y,
            s=45,
            zorder=5,
        )

        ax.annotate(
            note["text"],
            xy=(x, y),
            xytext=note["xytext"],
            textcoords="offset points",
            fontsize=9,
            arrowprops=dict(
                arrowstyle="->",
                lw=1.2,
            ),
            bbox=dict(
                boxstyle="round,pad=0.35",
                alpha=.60,
            ),
        )

    plt.tight_layout()
    plt.show()

Above we see the decomposition of the daily call volume over the last year. In the second subplot we see a trend that has a wave like shape (this is our summer-winter trend), in the third we see the seasonal component where the volume peaks in the middle of the week and dips during the weekends, and at the bottom we can see the residuals i.e points that were not captured well by the decomposition (in the residuals we see most hovering around 0). 

In [ ]:
day_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday",
]

daily_weekday_volume = (
    valid_time
    .copy()
    .assign(date=lambda df: pd.to_datetime(df["date"]))
    .groupby(["date", "day_of_week", "day_of_week_num"], as_index=False)
    .agg(
        unique_call_events=(EVENT_ID_COLUMN, "nunique"),
        dispatch_records=(ROW_ID_COLUMN, "nunique"),
    )
    .sort_values(["day_of_week_num", "date"])
)

fig = px.box(
    daily_weekday_volume,
    x="day_of_week",
    y="unique_call_events",
    points="outliers",
    title="Distribution of Daily SPD Call Events by Day of Week",
    labels={
        "day_of_week": "Day of week",
        "unique_call_events": "Unique call events per day",
    },
    category_orders={
        "day_of_week": day_order,
    },
)

fig.update_layout(
    xaxis_title="Day of week",
    yaxis_title="Unique call events per day",
    hovermode="closest",
)

# TAKE THIS OUT IF YOU DON"T LIKE IT DARK
fig.update_layout(
    template="plotly_dark",
    plot_bgcolor="#545455",   
    paper_bgcolor="#111111",
    )


fig.show()

daily_weekday_volume.head()

Above we see boxplots for each day of the week. This gives us insight into te typical values, typical ranges of values for each day of the week. Visually it is evident that Saturday and Sunday have lower call volumes but this could be a small enough difference to have just occurred by chance.

### Hourly Call Volume Analysis

Below we see boxplots for each hour of the day which gives us insight into the distributions of call volumes during these hours. It is evident that there is a lull during the hours most would consider nighttime. 

In [ ]:
# Build hourly call volume per date first, then compare those hourly values
# across all days in the dataset.
daily_hourly_volume = (
    valid_time
    .copy()
    .assign(
        date=lambda df: pd.to_datetime(df["date"]),
        hour=lambda df: df["hour"].astype(int),
    )
    .groupby(["date", "hour"], as_index=False)
    .agg(
        unique_call_events=(EVENT_ID_COLUMN, "nunique"),
        dispatch_records=(ROW_ID_COLUMN, "nunique"),
    )
)

# Optional but useful: make sure every date has all 24 hours represented.
# Missing date-hour combinations are treated as 0 calls.
all_dates = daily_hourly_volume["date"].drop_duplicates().sort_values()
all_hours = range(24)

full_date_hour_index = pd.MultiIndex.from_product(
    [all_dates, all_hours],
    names=["date", "hour"],
)

daily_hourly_volume = (
    daily_hourly_volume
    .set_index(["date", "hour"])
    .reindex(full_date_hour_index, fill_value=0)
    .reset_index()
)

# Box plot: distribution of unique call events by hour of day
fig = px.box(
    daily_hourly_volume,
    x="hour",
    y="unique_call_events",
    points="outliers",
    title="Distribution of Hourly SPD Call Events by Hour of Day",
    labels={
        "hour": "Hour of day",
        "unique_call_events": "Unique call events per hour",
    },
)

fig.update_layout(
    xaxis_title="Hour of day",
    yaxis_title="Unique call events per hour",
    hovermode="closest",
)

fig.update_xaxes(
    tickmode="array",
    tickvals=list(range(24)),
)

# TAKE THIS OUT IF YOU DON"T LIKE IT DARK
fig.update_layout(
    template="plotly_dark",
    plot_bgcolor="#545455",   
    paper_bgcolor="#111111",
    )


fig.show()


Below we see a plot that shows the boxplots for day-time and night-time hours and their call volume distributions. It is abundantly clear that nighttime hours see far less volume. An ANOVA could easily corroborate this. 

In [ ]:
day_night_data = (
    valid_time
    .copy()
    .assign(
        date=lambda df: pd.to_datetime(df["date"]),
        hour=lambda df: df["hour"].astype(int),
    )
)

day_night_data["time_period"] = np.where(
    (day_night_data["hour"] >= 21) | (day_night_data["hour"] < 7),
    "Nighttime: 9 PM–7 AM",
    "Daytime: 7 AM–9 PM",
)

day_night_data["period_date"] = day_night_data["date"]

# Assign after-midnight nighttime calls to the previous night
day_night_data.loc[
    day_night_data["hour"] < 7,
    "period_date"
] = day_night_data.loc[
    day_night_data["hour"] < 7,
    "period_date"
] - pd.Timedelta(days=1)

day_night_volume = (
    day_night_data
    .groupby(["period_date", "time_period"], as_index=False)
    .agg(
        unique_call_events=(EVENT_ID_COLUMN, "nunique"),
        dispatch_records=(ROW_ID_COLUMN, "nunique"),
    )
)

period_order = [
    "Daytime: 7 AM–9 PM",
    "Nighttime: 9 PM–7 AM",
]

fig = px.box(
    day_night_volume,
    x="time_period",
    y="unique_call_events",
    points="outliers",
    title="Distribution of SPD Call Events: Daytime vs Nighttime",
    labels={
        "time_period": "Time period",
        "unique_call_events": "Unique call events per period",
    },
    category_orders={
        "time_period": period_order,
    },
)

fig.update_layout(
    xaxis_title="Time period",
    yaxis_title="Unique call events per period",
    hovermode="closest",
)

# TAKE THIS OUT IF YOU DON"T LIKE IT DARK
fig.update_layout(
    template="plotly_dark",
    plot_bgcolor="#545455",   
    paper_bgcolor="#111111",
    )


fig.show()

Below we see a heatmap for the day of week and hour of day. You can see the phenomenon we talked about earlier where day time hours have higher call volumes, but the day of week has an impact on this. If we take a look at the Saturday and Sunday portion of the plot we can see that while there are not as many calls as there are during the peak of the daytime and during the week, there is still some activity from midnight to 2 am. We can also see that there is more activity during the hours from 9-12 on saturdays compared to Sundays. 

In [ ]:
hour_day_counts = (
    valid_time
    .groupby(["day_of_week", "day_of_week_num", "hour"])
    .agg(
        unique_call_events=(EVENT_ID_COLUMN, "nunique")
    )
    .reset_index()
)

hour_day_pivot = (
    hour_day_counts
    .pivot_table(
        index="day_of_week",
        columns="hour",
        values="unique_call_events",
        aggfunc="sum",
        fill_value=0,
    )
    .reindex(day_order)
)

hour_day_pivot

fig = px.imshow(
    hour_day_pivot,
    title="SPD Call Events by Day of Week and Hour",
    labels={
        "x": "Hour of day",
        "y": "Day of week",
        "color": "Unique call events",
    },
    aspect="auto",
)

# TAKE THIS OUT IF YOU DON"T LIKE IT DARK
fig.update_layout(
    template="plotly_dark",
    plot_bgcolor="#545455",   
    paper_bgcolor="#111111",
    )


fig.show()

Below we see the boxplots for summer vs non-summer months and their call volume distributions. There is some visual evidence that there is less call volume during the non-summer months, but this will likely need to be followed up with an ANOVA.

In [ ]:
# Build daily call volume first, then compare summer vs non-summer daily values
daily_season_volume = (
    valid_time
    .copy()
    .assign(date=lambda df: pd.to_datetime(df["date"]))
    .groupby("date", as_index=False)
    .agg(
        unique_call_events=(EVENT_ID_COLUMN, "nunique"),
        dispatch_records=(ROW_ID_COLUMN, "nunique"),
    )
)

daily_season_volume["month_num"] = daily_season_volume["date"].dt.month

daily_season_volume["season_group"] = daily_season_volume["month_num"].apply(
    lambda month: "Summer: June–August" if month in [6, 7, 8] else "Non-summer"
)

season_order = [
    "Summer: June–August",
    "Non-summer",
]

fig = px.box(
    daily_season_volume,
    x="season_group",
    y="unique_call_events",
    points="outliers",
    title="Distribution of Daily SPD Call Events: Summer vs Non-Summer",
    labels={
        "season_group": "Season group",
        "unique_call_events": "Unique call events per day",
    },
    category_orders={
        "season_group": season_order,
    },
)

fig.update_layout(
    xaxis_title="Season group",
    yaxis_title="Unique call events per day",
    hovermode="closest",
)

# TAKE THIS OUT IF YOU DON"T LIKE IT DARK
fig.update_layout(
    template="plotly_dark",
    plot_bgcolor="#545455",   
    paper_bgcolor="#111111",
    )


fig.show()

daily_season_volume.head()

This one gives us some insight into the prevalence of the priority levels after we define what they mean. There is a high volume of calls for priority levels 2 and 3, 2 being urgent calls that are not quite at the level of being life threatening, 3 being non-emergency routine calls like noise-complaints or disturbances.

In [ ]:
priority_hourly_full = pd.read_parquet((PROJECT_ROOT / 'data' / 'processed' / 'priority_hourly_full.parquet').as_posix())

priority_hourly_full["hour"] = priority_hourly_full["hour"].astype(int)

priority_hourly_full["priority_label"] = (
    priority_hourly_full["priority"]
    .astype(float)
    .astype(int)
    .astype(str)
)

priority_hourly_full["unique_call_events"] = (
    priority_hourly_full["unique_call_events"]
    .astype(int)
)

priority_descriptions = {
    '1':'Incidents posing an imminent threat to life',
    '2':'Urgent (non-life threatening)',
    '3':'Non-emergency (Routine)',
    '4':'Administrative calls, cold incidents (Low-Level)',
    '5':'Alternative/Telephone reporting',
    '7':'Officer-Initiated Activity',
    '9':'Lowest Urgency / Information only',
}

priority_hourly_full["priority_description"] = (
    priority_hourly_full["priority_label"]
    .map(priority_descriptions)
    .fillna("Unknown priority")
)

priority_hourly_full["priority_display"] = (
    "Priority "
    + priority_hourly_full["priority_label"]
    + ": "
    + priority_hourly_full["priority_description"]
)

priority_hourly_full = priority_hourly_full.sort_values(
    ["priority_label", "hour"]
).reset_index(drop=True)

priority_hourly_full.head()

priority_order = ["1", "2", "3", "4", "5", "7", "9"]

fig = px.line(
    priority_hourly_full,
    x="hour",
    y="unique_call_events",
    color="priority_display",
    markers=False,
    title="SPD Unique CAD Events by Hour and Priority, Full Dataset",
    labels={
        "hour": "Hour of day",
        "unique_call_events": "Unique CAD events",
        "priority_display": "Priority",
    },
)

fig.update_layout(
    hovermode="x unified",
    xaxis_title="Hour of day",
    yaxis_title="Unique CAD events",
    legend_title_text="Priority",
)

fig.update_xaxes(
    tickmode="array",
    tickvals=list(range(24)),
)

# TAKE THIS OUT IF YOU DON"T LIKE IT DARK
fig.update_layout(
    template="plotly_dark",
    plot_bgcolor="#545455",   
    paper_bgcolor="#111111",
    )

visible_groups = [
    "Priority 1: Incidents posing an imminent threat to life",
    "Priority 2: Urgent (non-life threatening)",
    "Priority 3: Non-emergency (Routine)",
]

for trace in fig.data:
    if trace.name not in visible_groups:
        trace.visible = "legendonly"

fig.show()

In [ ]:
priority_df = valid_time.dropna(subset=["priority"]).copy()

priority_df["priority_label"] = (
    priority_df["priority"]
    .astype("Int64")
    .astype(str)
)

priority_counts = (
    priority_df
    .groupby("priority_label")
    .agg(
        unique_call_events=(EVENT_ID_COLUMN, "nunique")
    )
    .reset_index()
    .sort_values("priority_label")
)

priority_counts

priority_counts["priority_description"] = (
    priority_counts["priority_label"]
    .map(priority_descriptions)
    .fillna("Unknown priority")
)


fig = px.bar(
    priority_counts,
    x="priority_label",
    y="unique_call_events",
    text='priority_description',
    title="SPD Call Events by Priority",
    labels={
        "priority_label": "Priority",
        "unique_call_events": "Unique call events",
    },
)

# TAKE THIS OUT IF YOU DON"T LIKE IT DARK
fig.update_layout(
    template="plotly_dark",
    plot_bgcolor="#545455",   
    paper_bgcolor="#111111",
    )


fig.show() 

#### Additional Visuals, Notes, Findings

In [ ]:
arson_df = monthly_by_event_group[
    monthly_by_event_group["event_group"].isin(["arson, bombs, explosion"])
].copy()

fig_arson = px.line(
    arson_df,
    x="period_start",
    y="event_count",
    color="event_group",
    markers=False,
    title="Monthly SPD Unique CAD Event Volume: Arson, Bombs, Explosion",
    labels={
        "period_start": "Month",
        "event_count": "Unique CAD Events",
        "event_group": "Event Group",
    },
)

fig_arson.update_layout(
    hovermode="x unified",
    xaxis_title="Month",
    yaxis_title="Unique CAD Events",
    legend_title_text="Event Group",
    template="plotly_dark",
    plot_bgcolor="#545455",
    paper_bgcolor="#111111",
)

row_2017_july = arson_df.loc[
    (arson_df["period_start"] - pd.Timestamp("2017-07-01")).abs().idxmin()
]

fig_arson.add_annotation(
    x=row_2017_july["period_start"],
    y=row_2017_july["event_count"],
    text="July spikes likely driven by fireworks calls",
    showarrow=True,
    arrowhead=2,
    ax=50,
    ay=-80,
    bgcolor="rgba(17,17,17,0.85)",
    bordercolor="white",
    borderwidth=1,
    font=dict(size=11, color="white"),
)

row_2025_july = arson_df.loc[
    (arson_df["period_start"] - pd.Timestamp("2025-07-01")).abs().idxmin()
]

fig_arson.add_annotation(
    x=row_2025_july["period_start"],
    y=row_2025_july["event_count"],
    text="Recent July spikes appear large",
    showarrow=True,
    arrowhead=2,
    ax=-70,
    ay=-80,
    bgcolor="rgba(17,17,17,0.85)",
    bordercolor="white",
    borderwidth=1,
    font=dict(size=11, color="white"),
)

fig_arson.show()

In [ ]:
arson_actual = arson_df.copy()
arson_actual["series"] = "Actual"

arson_july_ffill = arson_df.copy()
arson_july_ffill["series"] = "July forward-filled"

arson_july_ffill.loc[
    arson_july_ffill["period_start"].dt.month == 7,
    "event_count"
] = pd.NA

arson_july_ffill["event_count"] = (
    arson_july_ffill
    .sort_values("period_start")
    .groupby("event_group")["event_count"]
    .ffill()
)

arson_compare = pd.concat(
    [arson_actual, arson_july_ffill],
    ignore_index=True,
)

fig_arson_ffill = px.line(
    arson_compare,
    x="period_start",
    y="event_count",
    color="series",
    line_dash="event_group",
    markers=False,
    title="Arson/Bombs/Explosion Calls: Actual vs July Forward-Filled",
    labels={
        "period_start": "Month",
        "event_count": "Unique CAD Events",
        "series": "Series",
        "event_group": "Event Group",
    },
)

fig_arson_ffill.update_layout(
    hovermode="x unified",
    xaxis_title="Month",
    yaxis_title="Unique CAD Events",
    template="plotly_dark",
    plot_bgcolor="#545455",
    paper_bgcolor="#111111",
)

fig_arson_ffill.show()

In [ ]:
liquor_df = monthly_by_event_group[
    monthly_by_event_group['event_group'] == 'intoxication & liquor violations'
    ].copy()

fig_liquor = px.line(
    liquor_df,
    x="period_start",
    y="event_count",
    color="event_group",
    markers=False,
    title="Monthly SPD Unique CAD Event Volume: Intoxication and Liquor Violations",
    labels={
        "period_start": "Month",
        "event_count": "Unique CAD Events",
        "event_group": "Event Group",
    },
)

fig_liquor.update_layout(
    hovermode="x unified",
    xaxis_title="Month",
    yaxis_title="Unique CAD Events",
    legend_title_text="Event Group",
    template="plotly_dark",
    plot_bgcolor="#545455",
    paper_bgcolor="#111111",
)

row_2020_june = liquor_df.loc[
    (liquor_df["period_start"] - pd.Timestamp("2023-10-01")).abs().idxmin()
]

fig_liquor.add_annotation(
    x=row_2020_june["period_start"],
    y=row_2020_june["event_count"],
    text="Long-run decline continues after 2020",
    showarrow=True,
    arrowhead=2,
    ax=50,
    ay=-70,
    bgcolor="rgba(17,17,17,0.85)",
    bordercolor="white",
    borderwidth=1,
    font=dict(size=11, color="white"),
)

fig_liquor.show()